# Dataset Versioning Functionality - Real Dataset Demo

This notebook demonstrates Phase 2: Dataset Versioning using a **copy of the actual production dataset**.

**Dataset**: `notebooks/data/products.parquet` (copy from `output/products.parquet`)

**Manifest**: `notebooks/data/verification_manifest.json` (copy from `output/verification_manifest.json`)

## Features Demonstrated
1. Auto-versioning mode
2. Creating multiple versions
3. Listing version history
4. Viewing version metadata
5. Symlink management
6. Manual cleanup of old versions
7. Manifest versioning and hashing

In [ ]:
# Imports
from pathlib import Path

import pandas as pd
from loguru import logger

from src.pipeline.dataset_builder import DatasetBuilder
from src.storage.versioning import DatasetVersionManager

# Configure logger for notebook
logger.remove()
logger.add(
    lambda msg: print(msg, end=""), colorize=True, format="<level>{message}</level>"
)

## Setup: Paths and Initial State

In [ ]:
# Paths (using notebooks/data/ directory)
dataset_path = Path("notebooks/data/products.parquet")
manifest_path = Path("notebooks/data/verification_manifest.json")
output_dir = dataset_path.parent

print(f"Dataset path: {dataset_path}")
print(f"Manifest path: {manifest_path}")
print(f"Output directory: {output_dir}")
print(f"Dataset exists: {dataset_path.exists()}")
print(f"Manifest exists: {manifest_path.exists()}")

## Step 1: Create Version 1 with Auto-Versioning

Build the dataset with `--version-mode auto`. This will:
- Create `products_v1.parquet`
- Create `verification_manifest_v1.json`
- Create `dataset_metadata.json` with version history
- Create symlinks pointing to version 1

In [ ]:
# Build dataset with auto-versioning
builder1 = DatasetBuilder(
    manifest_path=manifest_path,
    output_path=dataset_path,
    version_mode="auto",
    force=True,
)

print("Building dataset with auto-versioning...\n")
success = builder1.build_and_save()
print(f"\n{'✓' if success else '✗'} Build result: {'SUCCESS' if success else 'FAILED'}")

## Step 2: Verify Version 1 was Created

In [ ]:
# Check created files
version_manager = DatasetVersionManager(output_dir)

print("📁 Created Files:")
print(f"  products_v1.parquet: {(output_dir / 'products_v1.parquet').exists()}")
print(
    f"  verification_manifest_v1.json: {(output_dir / 'verification_manifest_v1.json').exists()}"
)
print(f"  dataset_metadata.json: {(output_dir / 'dataset_metadata.json').exists()}")
print(f"  products.parquet (symlink): {(output_dir / 'products.parquet').exists()}")
print(
    f"  verification_manifest.json (symlink): {(output_dir / 'verification_manifest.json').exists()}"
)

# Check dataset content
df1 = pd.read_parquet(output_dir / "products_v1.parquet")
print(f"\n📊 Version 1 Dataset: {len(df1)} records")

## Step 3: View Version Metadata

In [ ]:
# Get version info
v1_info = version_manager.get_version_info(1)

print("📝 Version 1 Metadata:")
print(f"  Version: {v1_info['version']}")
print(f"  Timestamp: {v1_info['timestamp']}")
print(f"  Records: {v1_info['record_count']}")
print(f"  File: {v1_info['file']}")
print(f"  Manifest: {v1_info['manifest_file']}")
print(f"  Manifest Hash: {v1_info['manifest_hash']}")
print(f"  Append Mode: {v1_info['append_mode']}")
print(f"  Created By: {v1_info['created_by']}")

## Step 4: Create Version 2 with Append

Append to the existing dataset while using versioning. This will:
- Create `products_v2.parquet`
- Track merge statistics (records added/updated)
- Link to parent version (v1)
- Update symlinks to point to v2

In [ ]:
# Build with append and versioning
builder2 = DatasetBuilder(
    manifest_path=manifest_path,
    output_path=dataset_path,
    append=True,
    merge_strategy="update",
    version_mode="auto",
    force=True,
)

print("Building dataset with append + versioning...\n")
success = builder2.build_and_save()
print(f"\n{'✓' if success else '✗'} Build result: {'SUCCESS' if success else 'FAILED'}")

## Step 5: Compare Version 1 and Version 2

In [ ]:
# Load both versions
df1 = pd.read_parquet(output_dir / "products_v1.parquet")
df2 = pd.read_parquet(output_dir / "products_v2.parquet")

print("📊 Version Comparison:")
print(f"  Version 1: {len(df1)} records")
print(f"  Version 2: {len(df2)} records")
print(f"  Difference: {len(df2) - len(df1):+d} records")

# Get version 2 metadata
v2_info = version_manager.get_version_info(2)
print("\n📝 Version 2 Metadata:")
print(f"  Parent Version: {v2_info.get('parent_version', 'None')}")
print(f"  Append Mode: {v2_info['append_mode']}")
print(f"  Merge Strategy: {v2_info.get('merge_strategy', 'N/A')}")
print(f"  Records Added: {v2_info.get('records_added', 0)}")
print(f"  Records Updated: {v2_info.get('records_updated', 0)}")

## Step 6: Create Version 3 (Overwrite Mode)

Create another version without append mode (complete replacement).

In [ ]:
# Build without append
builder3 = DatasetBuilder(
    manifest_path=manifest_path,
    output_path=dataset_path,
    append=False,
    version_mode="auto",
    force=True,
)

print("Building dataset (overwrite mode) with versioning...\n")
success = builder3.build_and_save()
print(f"\n{'✓' if success else '✗'} Build result: {'SUCCESS' if success else 'FAILED'}")

# Verify version 3
v3_info = version_manager.get_version_info(3)
print("\n📝 Version 3 Metadata:")
print(f"  Parent Version: {v3_info.get('parent_version', 'None (overwrite mode)')}")
print(f"  Append Mode: {v3_info['append_mode']}")
print(f"  Records: {v3_info['record_count']}")

## Step 7: List All Versions

View complete version history.

In [ ]:
# List all versions
versions = version_manager.list_versions()

print(f"📊 Dataset Version History ({len(versions)} versions)\n")

for v in versions:
    print(f"Version {v['version']}:")
    print(f"  Timestamp: {v['timestamp']}")
    print(f"  Records: {v['record_count']}")
    print(f"  File: {v['file']}")
    print(f"  Manifest: {v['manifest_file']} ({v['manifest_hash'][:16]}...)")
    print(f"  Append Mode: {v.get('append_mode', False)}")

    if v.get("append_mode"):
        print(
            f"  Added: {v.get('records_added', 0)}, Updated: {v.get('records_updated', 0)}"
        )
        print(f"  Parent: v{v.get('parent_version', 'N/A')}")
    print()

print(f"📌 Current Version: {version_manager.get_current_version()}")

## Step 8: Verify Symlinks

Check that symlinks point to the current version (v3).

In [ ]:
# Check symlinks
dataset_symlink = output_dir / "products.parquet"
manifest_symlink = output_dir / "verification_manifest.json"

print("🔗 Symlink Status:")
print(f"  products.parquet exists: {dataset_symlink.exists()}")
print(f"  verification_manifest.json exists: {manifest_symlink.exists()}")

# Verify they point to v3 (current version)
if dataset_symlink.exists():
    df_symlink = pd.read_parquet(dataset_symlink)
    df_v3 = pd.read_parquet(output_dir / "products_v3.parquet")
    print(f"\n  Symlink points to current version: {len(df_symlink) == len(df_v3)}")
    print(f"  Symlink records: {len(df_symlink)}")
    print(f"  Version 3 records: {len(df_v3)}")

## Step 9: Test Manual Cleanup

Clean up old versions, keeping only the last 2.

In [ ]:
# Show versions before cleanup
print("📊 Before Cleanup:")
versions_before = version_manager.list_versions()
print(f"  Total versions: {len(versions_before)}")
print(f"  Versions: {[v['version'] for v in versions_before]}")

# Cleanup - keep last 2
print("\n🗑️  Cleaning up old versions (keeping last 2)...")
version_manager.cleanup_old_versions(2)

# Show versions after cleanup
print("\n📊 After Cleanup:")
versions_after = version_manager.list_versions()
print(f"  Total versions: {len(versions_after)}")
print(f"  Versions: {[v['version'] for v in versions_after]}")
print(f"\n  Deleted: {len(versions_before) - len(versions_after)} version(s)")

## Step 10: Verify Files After Cleanup

In [ ]:
# Check which files still exist
print("📁 File Existence After Cleanup:")
for i in range(1, 4):
    dataset_file = output_dir / f"products_v{i}.parquet"
    manifest_file = output_dir / f"verification_manifest_v{i}.json"

    print(f"\n  Version {i}:")
    print(f"    products_v{i}.parquet: {dataset_file.exists()}")
    print(f"    verification_manifest_v{i}.json: {manifest_file.exists()}")

## Step 11: Verify Manifest Hashing

Check that manifest hashes are computed correctly and can detect changes.

In [ ]:
# Get remaining versions
remaining_versions = version_manager.list_versions()

print("🔐 Manifest Hash Verification:")
for v in remaining_versions:
    manifest_file = output_dir / v["manifest_file"]

    if manifest_file.exists():
        # Recompute hash
        computed_hash = version_manager._compute_file_hash(manifest_file)
        stored_hash = v["manifest_hash"]

        print(f"\n  Version {v['version']}:")
        print(f"    Manifest: {v['manifest_file']}")
        print(f"    Stored Hash: {stored_hash[:32]}...")
        print(f"    Computed Hash: {computed_hash[:32]}...")
        print(
            f"    Match: {stored_hash == computed_hash} ✓"
            if stored_hash == computed_hash
            else "    Match: False ✗"
        )

## Step 12: Cleanup Test Files

Remove all versioned files and metadata to reset for future runs.

In [ ]:
# List all version-related files
print("🗑️  Cleaning up test files...")

files_to_remove = [
    "dataset_metadata.json",
    "products.parquet",  # symlink
    "verification_manifest.json",  # symlink
]

# Add versioned files
for i in range(1, 10):  # Check up to v10
    files_to_remove.append(f"products_v{i}.parquet")
    files_to_remove.append(f"verification_manifest_v{i}.json")

removed_count = 0
for filename in files_to_remove:
    file_path = output_dir / filename
    if file_path.exists():
        file_path.unlink()
        removed_count += 1
        print(f"  ✓ Removed: {filename}")

print(f"\n✓ Cleanup complete: {removed_count} files removed")

## Summary

This notebook demonstrated:

1. ✅ **Auto-versioning** - Automatic version number increments
2. ✅ **Version metadata** - Timestamps, record counts, manifest tracking
3. ✅ **Manifest versioning** - SHA-256 hashing for integrity verification
4. ✅ **Symlink management** - Automatic updates to point to current version
5. ✅ **Version history** - Complete lineage tracking with parent versions
6. ✅ **Append mode integration** - Tracks added/updated records per version
7. ✅ **Manual cleanup** - Keep last N versions, delete older ones
8. ✅ **File integrity** - Hash verification for manifests

### Key Features

**Version Structure:**
```
output/
├── products.parquet              # Symlink → products_v3.parquet
├── products_v2.parquet           # Version 2 snapshot
├── products_v3.parquet           # Version 3 snapshot (current)
├── verification_manifest.json    # Symlink → verification_manifest_v3.json
├── verification_manifest_v2.json # Version 2 manifest
├── verification_manifest_v3.json # Version 3 manifest
└── dataset_metadata.json         # Version history and metadata
```

**Metadata Schema:**
- Version number
- Timestamp (ISO 8601 with timezone)
- Record count
- Manifest file and SHA-256 hash
- Append mode flag
- Merge statistics (if append mode)
- Parent version (if append mode)

### CLI Quick Reference

```bash
# Build with auto-versioning
python build_dataset.py --version-mode auto

# Append with versioning
python build_dataset.py --append --version-mode auto

# List all versions
python build_dataset.py --list-versions

# Clean up old versions (keep last 3)
python build_dataset.py --cleanup-versions 3

# Manual version number
python build_dataset.py --version-mode manual --version 10
```